In [2]:
import numpy as np
import pickle
import os

In [3]:
edge_path = r"mydata/network_porto/porto_edges_new_simplify.pkl"
node_path = r"mydata/network_porto/porto_nodes_new.pkl"
output_path = r"examples/"
scaler_list = [
    "porto_scaler_attr.pkl",
    "porto_scaler_gps.pkl"
]
train_path = r"mydata/train.npy"

In [4]:
with open(edge_path, 'rb') as f:
    edges = pickle.load(f)
    
with open(node_path, 'rb') as f:
    nodes = pickle.load(f)
    
train_data = np.load(train_path, allow_pickle=True)

In [5]:
linkids = []
dateinfo = []
inds = []
for d in train_data[:50000]:
    linkids.append(np.asarray(d[1]))
    dateinfo.append(d[2:5])
    inds.append(d[0])
lens = np.asarray([len(k) for k in linkids], dtype=np.int16)    

def info(xs, date):
    infos = []
    length = 0
    for x in xs:
        info = edges[x]
        infot = []
        infot.append(info[1])
        infot.append(length)
        length += info[1]
        infot += list(date)
        try:
            infot += [
                nodes[info[2]][0],
                nodes[info[2]][1],
                nodes[info[3]][0],
                nodes[info[3]][1]
            ]
        except:
            print(info)
        infos.append(np.asarray(infot))
    return infos
con_links = np.concatenate([info(b, dateinfo[ind]) for ind, b in enumerate(linkids)], dtype='object')
print(con_links[:5])

[[137.21387277578526 0.0 4.0 347.0 790.0 -8.6582785 41.1737841 -8.6594178
  41.1728968]
 [101.69483486112611 137.21387277578526 4.0 347.0 790.0 -8.6594178
  41.1728968 -8.660199 41.1735969]
 [24.34049008634803 238.90870763691137 4.0 347.0 790.0 -8.660199
  41.1735969 -8.6604891 41.1735816]
 [80.24928583999748 263.2491977232594 4.0 347.0 790.0 -8.6604891
  41.1735816 -8.6604467 41.1728607]
 [358.73014632145737 343.49848356325685 4.0 347.0 790.0 -8.6604467
  41.1728607 -8.6575202 41.1706814]]


In [6]:
from sklearn.preprocessing import StandardScaler
from utils.util import StandardScaler2

In [7]:
print(con_links.shape)

(2378874, 9)


In [8]:
attr_data = con_links[:,0:2]
gps_data = con_links[:,5:9]

scaler_std = StandardScaler().fit(attr_data)
scaler_std2 = StandardScaler().fit(gps_data)

In [9]:
print("=== StandardScaler (cols 0:2) ===")
print("Mean :", scaler_std.mean_)
print("Scale:", scaler_std.scale_)

print("\n=== StandardScaler2 (cols 5:9) ===")
# assuming StandardScaler2 exposes similar attributes
print("Mean :", scaler_std2.mean_)
print("Scale:", scaler_std2.scale_)

=== StandardScaler (cols 0:2) ===
Mean : [ 103.12735139 3045.15845699]
Scale: [ 119.92430708 2850.4650745 ]

=== StandardScaler2 (cols 5:9) ===
Mean : [-8.61921767 41.15787444 -8.6192473  41.15791222]
Scale: [0.02390028 0.00976221 0.02391369 0.00975475]


In [10]:
with open(os.path.join(output_path,scaler_list[0]), 'wb') as f:
    pickle.dump(scaler_std, f)

with open(os.path.join(output_path,scaler_list[1]), 'wb') as f:
    pickle.dump(scaler_std2, f)